In [1]:
import re, ftfy

def read_github_csv(url, encoding="latin1"):
    import requests
    from io import StringIO
    import pandas as pd
    
    text = requests.get(url).content.decode(encoding)
    return pd.read_csv(StringIO(text))

In [2]:


url = "https://raw.githubusercontent.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection/master/re_dataset.csv"

df = read_github_csv(url)
df

,Tweet,HS,Abusive,HS_Individual,HS_Group,HS_Religion,HS_Race,HS_Physical,HS_Gender,HS_Other,HS_Weak,HS_Moderate,HS_Strong
0,- disaat semua cowok berusaha melacak perhatia...,1,1,1,0,0,0,0,0,1,1,0,0
1,RT USER: USER siapa yang telat ngasih tau elu?...,0,1,0,0,0,0,0,0,0,0,0,0
2,"41. Kadang aku berfikir, kenapa aku tetap perc...",0,0,0,0,0,0,0,0,0,0,0,0
3,USER USER AKU ITU AKU\n\nKU TAU MATAMU SIPIT T...,0,0,0,0,0,0,0,0,0,0,0,0
4,USER USER Kaum cebong kapir udah keliatan dong...,1,1,0,1,1,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13164,USER jangan asal ngomong ndasmu. congor lu yg ...,1,1,1,0,0,0,1,0,0,1,0,0
13165,USER Kasur mana enak kunyuk',0,1,0,0,0,0,0,0,0,0,0,0
13166,USER Hati hati bisu :( .g\n\nlagi bosan huft \...,0,0,0,0,0,0,0,0,0,0,0,0
13167,USER USER USER USER Bom yang real mudah terdet...,0,0,0,0,0,0,0,0,0,0,0,0


### Text Cleaning

In [3]:
def fix_text(text):
    return ftfy.fix_text(text)

def remove_newline(text):
    return text.replace("\n", " ")

def remove_emoji(text):
    emoji_pattern = re.compile("[" 
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub("", text)

def remove_hex_emoji(text):
    # Hapus pola emoji hex seperti \xf0\x9f\x98\x84
    text = re.sub(r"\\x[a-fA-F0-9]{2}", " ", text)
    text = re.sub(r"x[a-fA-F0-9]{2}", " ", text)
    return text

def to_lower(text):
    return text.lower()

def remove_user_url(text):
    text = re.sub(r"\bUSER\b", " ", text)
    text = re.sub(r"\bURL\b", " ", text)
    return text

def remove_rt(text):
    return re.sub(r"\brt\b", " ", text)

def remove_thread_number(text):
    return re.sub(r"^\s*\d+\.\s*", "", text)

def remove_escape_sequences(text):
    text = re.sub(r"\\[nrtvf0]", " ", text)  # hapus \n, \t, \r, dll.
    text = re.sub(r"\bn\b", " ", text)       # hapus 'n' sisa escape literal
    return text

def clean_symbols(text):
    # HAPUS noise, SIMPAN .,!?-
    return re.sub(r"[^0-9a-zA-Z\s.,!?-]", " ", text)

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

# FINAL PIPELINE
def preprocess_text(text):
    text = fix_text(text)
    text = remove_newline(text)
    text = remove_user_url(text)
    text = to_lower(text)
    text = remove_emoji(text)
    text = remove_hex_emoji(text)
    text = remove_rt(text)
    text = remove_thread_number(text)
    text = remove_escape_sequences(text)
    text = clean_symbols(text)
    text = normalize_whitespace(text)
    return text

In [4]:
df["clean_text"] = df["Tweet"].apply(preprocess_text)
df

,Tweet,HS,Abusive,HS_Individual,HS_Group,HS_Religion,HS_Race,HS_Physical,HS_Gender,HS_Other,HS_Weak,HS_Moderate,HS_Strong,clean_text
0,- disaat semua cowok berusaha melacak perhatia...,1,1,1,0,0,0,0,0,1,1,0,0,- disaat semua cowok berusaha melacak perhatia...
1,RT USER: USER siapa yang telat ngasih tau elu?...,0,1,0,0,0,0,0,0,0,0,0,0,siapa yang telat ngasih tau elu?edan sarap gue...
2,"41. Kadang aku berfikir, kenapa aku tetap perc...",0,0,0,0,0,0,0,0,0,0,0,0,"kadang aku berfikir, kenapa aku tetap percaya ..."
3,USER USER AKU ITU AKU\n\nKU TAU MATAMU SIPIT T...,0,0,0,0,0,0,0,0,0,0,0,0,aku itu aku ku tau matamu sipit tapi diliat da...
4,USER USER Kaum cebong kapir udah keliatan dong...,1,1,0,1,1,0,0,0,0,0,1,0,kaum cebong kapir udah keliatan dongoknya dari...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13164,USER jangan asal ngomong ndasmu. congor lu yg ...,1,1,1,0,0,0,1,0,0,1,0,0,jangan asal ngomong ndasmu. congor lu yg sekat...
13165,USER Kasur mana enak kunyuk',0,1,0,0,0,0,0,0,0,0,0,0,kasur mana enak kunyuk
13166,USER Hati hati bisu :( .g\n\nlagi bosan huft \...,0,0,0,0,0,0,0,0,0,0,0,0,hati hati bisu .g lagi bosan huft
13167,USER USER USER USER Bom yang real mudah terdet...,0,0,0,0,0,0,0,0,0,0,0,0,bom yang real mudah terdeteksi bom yang terkub...


### Penanganan Duplikasi Data

In [5]:
df['clean_text'].duplicated().sum()

np.int64(287)

In [6]:
df[df['clean_text'].duplicated(keep='first')]

,Tweet,HS,Abusive,HS_Individual,HS_Group,HS_Religion,HS_Race,HS_Physical,HS_Gender,HS_Other,HS_Weak,HS_Moderate,HS_Strong,clean_text
288,USER USER USER USER USER USER USER USER USER U...,1,1,0,1,0,0,0,0,1,0,1,0,
295,#GubernurZamanNow #GusIpulPuti2 #GanjarYasin1 ...,0,0,0,0,0,0,0,0,0,0,0,0,gubernurzamannow gusipulputi2 ganjaryasin1 dja...
318,USER USER USER USER USER USER USER USER USER U...,0,0,0,0,0,0,0,0,0,0,0,0,
377,USER USER USER USER USER USER USER USER USER U...,0,0,0,0,0,0,0,0,0,0,0,0,
378,#GubernurZamanNow #GusIpulPuti2 #GanjarYasin1 ...,0,0,0,0,0,0,0,0,0,0,0,0,gubernurzamannow gusipulputi2 ganjaryasin1 dja...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13054,nih ya baca tai. udah dong jangan spam minta k...,1,1,1,0,0,0,0,0,1,1,0,0,nih ya baca tai. udah dong jangan spam minta k...
13075,*Bravoo Presiden Jokowi !ð???* Presiden Joko W...,0,0,0,0,0,0,0,0,0,0,0,0,bravoo presiden jokowi ! ??? presiden joko wid...
13085,;Kebanyakan fitnah nih mpok silvy #DebatFinalP...,1,0,1,0,0,0,0,0,1,1,0,0,kebanyakan fitnah nih mpok silvy debatfinalpil...
13087,Ngentot Diatas\nURL,0,1,0,0,0,0,0,0,0,0,0,0,ngentot diatas url


In [7]:
df = df.drop_duplicates(subset='clean_text', keep='first')
df

,Tweet,HS,Abusive,HS_Individual,HS_Group,HS_Religion,HS_Race,HS_Physical,HS_Gender,HS_Other,HS_Weak,HS_Moderate,HS_Strong,clean_text
0,- disaat semua cowok berusaha melacak perhatia...,1,1,1,0,0,0,0,0,1,1,0,0,- disaat semua cowok berusaha melacak perhatia...
1,RT USER: USER siapa yang telat ngasih tau elu?...,0,1,0,0,0,0,0,0,0,0,0,0,siapa yang telat ngasih tau elu?edan sarap gue...
2,"41. Kadang aku berfikir, kenapa aku tetap perc...",0,0,0,0,0,0,0,0,0,0,0,0,"kadang aku berfikir, kenapa aku tetap percaya ..."
3,USER USER AKU ITU AKU\n\nKU TAU MATAMU SIPIT T...,0,0,0,0,0,0,0,0,0,0,0,0,aku itu aku ku tau matamu sipit tapi diliat da...
4,USER USER Kaum cebong kapir udah keliatan dong...,1,1,0,1,1,0,0,0,0,0,1,0,kaum cebong kapir udah keliatan dongoknya dari...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13164,USER jangan asal ngomong ndasmu. congor lu yg ...,1,1,1,0,0,0,1,0,0,1,0,0,jangan asal ngomong ndasmu. congor lu yg sekat...
13165,USER Kasur mana enak kunyuk',0,1,0,0,0,0,0,0,0,0,0,0,kasur mana enak kunyuk
13166,USER Hati hati bisu :( .g\n\nlagi bosan huft \...,0,0,0,0,0,0,0,0,0,0,0,0,hati hati bisu .g lagi bosan huft
13167,USER USER USER USER Bom yang real mudah terdet...,0,0,0,0,0,0,0,0,0,0,0,0,bom yang real mudah terdeteksi bom yang terkub...


### Reset Index

In [8]:
df.reset_index(inplace=True, drop=True)
df

,Tweet,HS,Abusive,HS_Individual,HS_Group,HS_Religion,HS_Race,HS_Physical,HS_Gender,HS_Other,HS_Weak,HS_Moderate,HS_Strong,clean_text
0,- disaat semua cowok berusaha melacak perhatia...,1,1,1,0,0,0,0,0,1,1,0,0,- disaat semua cowok berusaha melacak perhatia...
1,RT USER: USER siapa yang telat ngasih tau elu?...,0,1,0,0,0,0,0,0,0,0,0,0,siapa yang telat ngasih tau elu?edan sarap gue...
2,"41. Kadang aku berfikir, kenapa aku tetap perc...",0,0,0,0,0,0,0,0,0,0,0,0,"kadang aku berfikir, kenapa aku tetap percaya ..."
3,USER USER AKU ITU AKU\n\nKU TAU MATAMU SIPIT T...,0,0,0,0,0,0,0,0,0,0,0,0,aku itu aku ku tau matamu sipit tapi diliat da...
4,USER USER Kaum cebong kapir udah keliatan dong...,1,1,0,1,1,0,0,0,0,0,1,0,kaum cebong kapir udah keliatan dongoknya dari...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12877,USER jangan asal ngomong ndasmu. congor lu yg ...,1,1,1,0,0,0,1,0,0,1,0,0,jangan asal ngomong ndasmu. congor lu yg sekat...
12878,USER Kasur mana enak kunyuk',0,1,0,0,0,0,0,0,0,0,0,0,kasur mana enak kunyuk
12879,USER Hati hati bisu :( .g\n\nlagi bosan huft \...,0,0,0,0,0,0,0,0,0,0,0,0,hati hati bisu .g lagi bosan huft
12880,USER USER USER USER Bom yang real mudah terdet...,0,0,0,0,0,0,0,0,0,0,0,0,bom yang real mudah terdeteksi bom yang terkub...


### Slang

In [9]:
alay_df = read_github_csv(
    "https://raw.githubusercontent.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection/master/new_kamusalay.csv",
    encoding="latin1"
)

# jika hanya 1 kolom, pecah sendiri
if alay_df.shape[1] == 1:
    alay_df = alay_df[0].str.split(",", expand=True)

alay_df.columns = [0, 1]
alay_dict = dict(zip(alay_df[0], alay_df[1]))
alay_dict


{'pakcikdahtua': 'pak cik sudah tua',
 'pakcikmudalagi': 'pak cik muda lagi',
 't3tapjokowi': 'tetap jokowi',
 '3x': 'tiga kali',
 'aamiin': 'amin',
 'aamiinn': 'amin',
 'aamin': 'amin',
 'aammiin': 'amin',
 'abis': 'habis',
 'abisin': 'habiskan',
 'acau': 'kacau',
 'achok': 'ahok',
 'ad': 'ada',
 'adek': 'adik',
 'adl': 'adalah',
 'adlah': 'adalah',
 'adlh': 'adalah',
 'ado': 'ada',
 'aduhh': 'aduh',
 'aer': 'air',
 'afdol': 'afdal',
 'agamaataualqur': 'agama alquran',
 'agm': 'agama',
 'agma': 'agama',
 'ahaha': 'haha',
 'ahahaha': 'haha',
 'ahehehehe': 'hehe',
 'ahir': 'akhir',
 'ahirnya': 'akhirnya',
 'ahk': 'ahok',
 'ahlamdulillah': 'alhamdulillah',
 'ahli2': 'para ahli',
 'ahlusunnah': 'ahlus sunah',
 'ahmaddani': 'ahmad dhani',
 'aho': 'ahok',
 'ahoax': 'ahok',
 'ahoaxx': 'ahok',
 'ahog': 'ahok',
 'ahokataudjarot': 'ahok djarot',
 'ahokbebanijokowi': 'ahok beban jokowi',
 'ahokbtp': 'ahok basuki tjahaja purnama',
 'ahokditolakwarga': 'ahok ditolak warga',
 'ahokdjarot': 'ahok dj

In [10]:
# def normalize_slang(text):
#     return " ".join([alay_dict.get(w, w) for w in text.split()])

In [11]:
# df = df.copy()
# df['normalized_comment'] = df['clean_text'].apply(normalize_slang)
# df[['Tweet', 'normalized_comment']].head(50)

In [12]:
import requests

# 1. Loader kamus dari GitHub RAW

def load_dict_from_github(url):
    d = {}
    text = requests.get(url).text.splitlines()
    for line in text:
        if ":" in line:
            k, v = line.split(":", 1)
            d[k] = v
    return d

# 2. Load slang dan slur

slang_url = "https://raw.githubusercontent.com/Alfa4026/Skripsi/main/kamus/slang_formal_colon.txt"
slur_url  = "https://raw.githubusercontent.com/Alfa4026/Skripsi/main/kamus/slur_normalization_colon.txt"

slang_dict = load_dict_from_github(slang_url)
slur_dict  = load_dict_from_github(slur_url)

# 3. Fungsi NORMALISASI final

def normalize_slang_and_slur(text):
    new_words = []
    for w in text.split():
        w = w.strip()

        # 1. normalisasi slur (anjg → anjing)
        if w in slur_dict:
            new_words.append(slur_dict[w])
            continue

        # 2. normalisasi slang (gk → tidak, gue → saya)
        if w in slang_dict:
            new_words.append(slang_dict[w])
            continue

        # 3. token tetap
        new_words.append(w)

    return " ".join(new_words)


In [13]:
df = df.copy()
df['normalized_comment'] = df['clean_text'].apply(normalize_slang_and_slur)
df[['Tweet', 'clean_text', 'normalized_comment']]

,Tweet,clean_text,normalized_comment
0,- disaat semua cowok berusaha melacak perhatia...,- disaat semua cowok berusaha melacak perhatia...,- disaat semua cowok berusaha melacak perhatia...
1,RT USER: USER siapa yang telat ngasih tau elu?...,siapa yang telat ngasih tau elu?edan sarap gue...,siapa yang telat ngasih tau elu?edan sarap say...
2,"41. Kadang aku berfikir, kenapa aku tetap perc...","kadang aku berfikir, kenapa aku tetap percaya ...","kadang aku berfikir, kenapa aku tetap percaya ..."
3,USER USER AKU ITU AKU\n\nKU TAU MATAMU SIPIT T...,aku itu aku ku tau matamu sipit tapi diliat da...,aku itu aku ku tau matamu sipit tapi diliat da...
4,USER USER Kaum cebong kapir udah keliatan dong...,kaum cebong kapir udah keliatan dongoknya dari...,kaum cebong kapir udah keliatan dongoknya dari...
...,...,...,...
12877,USER jangan asal ngomong ndasmu. congor lu yg ...,jangan asal ngomong ndasmu. congor lu yg sekat...,jangan asal ngomong ndasmu. congor lu yang sek...
12878,USER Kasur mana enak kunyuk',kasur mana enak kunyuk,kasur mana enak kunyuk
12879,USER Hati hati bisu :( .g\n\nlagi bosan huft \...,hati hati bisu .g lagi bosan huft,hati hati bisu .g lagi bosan huft
12880,USER USER USER USER Bom yang real mudah terdet...,bom yang real mudah terdeteksi bom yang terkub...,bom yang real mudah terdeteksi bom yang terkub...


In [14]:
from collections import Counter

all_words = " ".join(df['normalized_comment'].astype(str)).split()
Counter(all_words).most_common(1000)


[('yang', 4885),
 ('tidak', 2718),
 ('dan', 2674),
 ('di', 2668),
 ('saya', 1681),
 ('itu', 1521),
 ('ada', 1292),
 ('ini', 1262),
 ('kalau', 1246),
 ('jadi', 1045),
 ('dengan', 995),
 ('orang', 982),
 ('saja', 895),
 ('tapi', 869),
 ('dari', 819),
 ('untuk', 808),
 ('juga', 798),
 ('bisa', 794),
 ('jokowi', 787),
 ('sama', 784),
 ('kamu', 772),
 ('ingin', 760),
 ('presiden', 758),
 ('aku', 705),
 ('ke', 668),
 ('ya', 666),
 ('dia', 619),
 ('kita', 618),
 ('sudah', 606),
 ('islam', 579),
 ('apa', 562),
 ('agama', 558),
 ('karena', 547),
 ('indonesia', 539),
 ('dalam', 518),
 ('bukan', 517),
 ('pak', 493),
 ('akan', 491),
 ('nya', 474),
 ('lu', 450),
 ('?', 446),
 ('lagi', 443),
 ('si', 442),
 ('tak', 438),
 ('cebong', 436),
 ('.', 425),
 ('pada', 401),
 ('atau', 390),
 ('udah', 388),
 ('mereka', 382),
 ('lebih', 381),
 ('adalah', 377),
 ('-', 368),
 ('banyak', 358),
 (',', 349),
 ('gubernur', 348),
 ('buat', 344),
 ('semua', 341),
 ('cina', 340),
 ('jangan', 339),
 ('banget', 336),
 ('

In [21]:
import requests

def load_set_from_github(url):
    return {line.strip() for line in requests.get(url).text.splitlines()}

stopwords_url = "https://raw.githubusercontent.com/Alfa4026/Skripsi/main/kamus/stopwords_colon.txt"
removal_url   = "https://raw.githubusercontent.com/Alfa4026/Skripsi/main/kamus/removal_tokens_colon.txt"

stopwords_set  = load_set_from_github(stopwords_url)
removal_tokens = load_set_from_github(removal_url)


In [22]:
def clean_stopword(text):
    words = text.split()
    cleaned = []

    for w in words:

        # 1. Hapus token noise
        if w in removal_tokens:
            continue

        # 2. Hapus stopword
        if w in stopwords_set:
            continue

        cleaned.append(w)

    return " ".join(cleaned)


In [23]:
df = df.copy()

df['stopword_clean'] = df['normalized_comment'].apply(clean_stopword)

df[['Tweet', 'clean_text', 'normalized_comment', 'stopword_clean']]


,Tweet,clean_text,normalized_comment,stopword_clean
0,- disaat semua cowok berusaha melacak perhatia...,- disaat semua cowok berusaha melacak perhatia...,- disaat semua cowok berusaha melacak perhatia...,disaat cowok berusaha melacak perhatian gue. k...
1,RT USER: USER siapa yang telat ngasih tau elu?...,siapa yang telat ngasih tau elu?edan sarap gue...,siapa yang telat ngasih tau elu?edan sarap say...,siapa telat ngasih tau elu?edan sarap saya ber...
2,"41. Kadang aku berfikir, kenapa aku tetap perc...","kadang aku berfikir, kenapa aku tetap percaya ...","kadang aku berfikir, kenapa aku tetap percaya ...","kadang aku berfikir, kenapa aku tetap percaya ..."
3,USER USER AKU ITU AKU\n\nKU TAU MATAMU SIPIT T...,aku itu aku ku tau matamu sipit tapi diliat da...,aku itu aku ku tau matamu sipit tapi diliat da...,aku aku tau matamu sipit tapi diliat mana aku
4,USER USER Kaum cebong kapir udah keliatan dong...,kaum cebong kapir udah keliatan dongoknya dari...,kaum cebong kapir udah keliatan dongoknya dari...,kaum cebong kapir udah keliatan dongoknya awal...
...,...,...,...,...
12877,USER jangan asal ngomong ndasmu. congor lu yg ...,jangan asal ngomong ndasmu. congor lu yg sekat...,jangan asal ngomong ndasmu. congor lu yang sek...,jangan asal ngomong ndasmu. congor lu sekate2 ...
12878,USER Kasur mana enak kunyuk',kasur mana enak kunyuk,kasur mana enak kunyuk,kasur mana enak kunyuk
12879,USER Hati hati bisu :( .g\n\nlagi bosan huft \...,hati hati bisu .g lagi bosan huft,hati hati bisu .g lagi bosan huft,hati hati bisu .g bosan huft
12880,USER USER USER USER Bom yang real mudah terdet...,bom yang real mudah terdeteksi bom yang terkub...,bom yang real mudah terdeteksi bom yang terkub...,bom real mudah terdeteksi bom terkubur suatu s...


In [18]:
df

,Tweet,HS,Abusive,HS_Individual,HS_Group,HS_Religion,HS_Race,HS_Physical,HS_Gender,HS_Other,HS_Weak,HS_Moderate,HS_Strong,clean_text,normalized_comment,stopword_clean
0,- disaat semua cowok berusaha melacak perhatia...,1,1,1,0,0,0,0,0,1,1,0,0,- disaat semua cowok berusaha melacak perhatia...,- disaat semua cowok berusaha melacak perhatia...,disaat cowok berusaha melacak perhatian gue. k...
1,RT USER: USER siapa yang telat ngasih tau elu?...,0,1,0,0,0,0,0,0,0,0,0,0,siapa yang telat ngasih tau elu?edan sarap gue...,siapa yang telat ngasih tau elu?edan sarap say...,siapa telat ngasih tau elu?edan sarap saya ber...
2,"41. Kadang aku berfikir, kenapa aku tetap perc...",0,0,0,0,0,0,0,0,0,0,0,0,"kadang aku berfikir, kenapa aku tetap percaya ...","kadang aku berfikir, kenapa aku tetap percaya ...","kadang aku berfikir, kenapa aku tetap percaya ..."
3,USER USER AKU ITU AKU\n\nKU TAU MATAMU SIPIT T...,0,0,0,0,0,0,0,0,0,0,0,0,aku itu aku ku tau matamu sipit tapi diliat da...,aku itu aku ku tau matamu sipit tapi diliat da...,aku aku tau matamu sipit tapi diliat mana aku
4,USER USER Kaum cebong kapir udah keliatan dong...,1,1,0,1,1,0,0,0,0,0,1,0,kaum cebong kapir udah keliatan dongoknya dari...,kaum cebong kapir udah keliatan dongoknya dari...,kaum cebong kapir udah keliatan dongoknya awal...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12877,USER jangan asal ngomong ndasmu. congor lu yg ...,1,1,1,0,0,0,1,0,0,1,0,0,jangan asal ngomong ndasmu. congor lu yg sekat...,jangan asal ngomong ndasmu. congor lu yang sek...,jangan asal ngomong ndasmu. congor lu sekate2 ...
12878,USER Kasur mana enak kunyuk',0,1,0,0,0,0,0,0,0,0,0,0,kasur mana enak kunyuk,kasur mana enak kunyuk,kasur mana enak kunyuk
12879,USER Hati hati bisu :( .g\n\nlagi bosan huft \...,0,0,0,0,0,0,0,0,0,0,0,0,hati hati bisu .g lagi bosan huft,hati hati bisu .g lagi bosan huft,hati hati bisu .g bosan huft
12880,USER USER USER USER Bom yang real mudah terdet...,0,0,0,0,0,0,0,0,0,0,0,0,bom yang real mudah terdeteksi bom yang terkub...,bom yang real mudah terdeteksi bom yang terkub...,bom real mudah terdeteksi bom terkubur suatu s...


In [19]:
# df.to_csv("hatespeech_preprocessed.csv", index=False)

In [24]:
df["stopword_clean"].str.split(expand=True).stack().value_counts().head(300)


tidak         2718
saya          1681
ada           1292
kalau         1246
orang          982
              ... 
dki             71
indonesia.      71
berita          71
bakal           71
beliau          71
Name: count, Length: 300, dtype: int64